In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 60.5 MB/s eta 0:00:00


In [ ]:
pip install joblib

In [ ]:
import numpy as np
import pickle
from gensim.models import Word2Vec
import joblib

save_dir = "/content/drive/MyDrive/NLP_Project_Preprocessing"

X_train_pad = np.load(f"{save_dir}/X_train_pad.npy")
X_test_pad = np.load(f"{save_dir}/X_test_pad.npy")

y_train_sentiment = np.load(f"{save_dir}/y_train_sentiment.npy")
y_test_sentiment = np.load(f"{save_dir}/y_test_sentiment.npy")

embedding_matrix = np.load(f"{save_dir}/embedding_matrix.npy")

with open(f"{save_dir}/word_index.pkl", "rb") as f:
    word_index = pickle.load(f)

with open(f"{save_dir}/sentiment_encoder.pkl", "rb") as f:
    sentiment_encoder = joblib.load(f)


word2vec_model = Word2Vec.load(f"{save_dir}/word2vec.model")

vocab_size = len(word_index) + 1
embedding_dim = embedding_matrix.shape[1]
max_len = X_train_pad.shape[1]

print(f"Training samples : {X_train_pad.shape[0]:,}")
print(f"Test samples     : {X_test_pad.shape[0]:,}")
print(f"Vocabulary size  : {vocab_size:,}")
print(f"Embedding dim    : {embedding_dim}")
print(f"Sequence length  : {max_len}")
print(f"Classes          : {len(sentiment_encoder.classes_)}")
print("Labels:", sentiment_encoder.classes_)

Training samples : 1,166,475
Test samples     : 291,619
Vocabulary size  : 81,957
Embedding dim    : 100
Sequence length  : 30
Classes          : 3
Labels: ['negative' 'neutral' 'positive']


In [ ]:
pip install keras-tcn

INFO: pip is looking at multiple versions of keras-tcn to determine which version is compatible with other requirements. This could take a while.


In [ ]:
from tcn import TCN

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import SpatialDropout1D
from tensorflow.keras.optimizers import Adam
from tcn import TCN

model = Sequential([
    Embedding(
        input_dim=embedding_matrix.shape[0],
        output_dim=embedding_matrix.shape[1],
        weights=[embedding_matrix],
        trainable=False
    ),

    SpatialDropout1D(0.2),

    TCN(
        nb_filters=64,
        kernel_size=3,
        dilations=[1,2,4,8],
        dropout_rate=0.2,
        return_sequences=False
    ),

    Dense(32, activation='relu'),

    Dropout(0.5),

    Dense(3, activation='softmax')
])

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

In [ ]:
model.compile(
    optimizer="adam",
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train_sentiment)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_sentiment
)

class_weights = dict(zip(classes, weights))
print(class_weights)

{np.int64(0): np.float64(0.8395355214264277), np.int64(1): np.float64(0.5681477790620028), np.int64(2): np.float64(20.508729363363045)}


In [ ]:
history = model.fit(
    X_train_pad,
    y_train_sentiment,
    validation_split=0.1,
    epochs=10,
    batch_size=128,
    class_weight=class_weights,
    callbacks=[early_stopping]
)

Epoch 1/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 103s 11ms/step - accuracy: 0.5017 - loss: 0.8353 - val_accuracy: 0.6231 - val_loss: 0.7859
Epoch 2/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 103s 7ms/step - accuracy: 0.5962 - loss: 0.7186 - val_accuracy: 0.6546 - val_loss: 0.7083
Epoch 3/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 86s 8ms/step - accuracy: 0.6155 - loss: 0.6747 - val_accuracy: 0.6536 - val_loss: 0.7405
Epoch 4/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 52s 6ms/step - accuracy: 0.6268 - loss: 0.6566 - val_accuracy: 0.6790 - val_loss: 0.7018
Epoch 5/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 79s 6ms/step - accuracy: 0.6335 - loss: 0.6479 - val_accuracy: 0.6718 - val_loss: 0.6876
Epoch 6/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 49s 6ms/step - accuracy: 0.6377 - loss: 0.6380 - val_accuracy: 0.7145 - val_loss: 0.6250
Epoch 7/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 48s 6ms/step - accuracy: 0.6413 - loss: 0.6345 - val_accuracy: 0.6865 - val_loss: 0.6623
Epoch 8/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 48s 6ms/step - accuracy: 0.6389 - loss

In [ ]:
import numpy as np

y_pred_probs = model.predict(X_test_pad)

y_pred = np.argmax(y_pred_probs, axis=1)

9114/9114 ━━━━━━━━━━━━━━━━━━━━ 23s 2ms/step


In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test_sentiment,
        y_pred,
        target_names=[
            "Negative",
            "Neutral",
            "Positive"
        ]
    )
)

              precision    recall  f1-score   support

    Negative       0.71      0.76      0.74    115785
     Neutral       0.82      0.68      0.75    171094
    Positive       0.15      0.87      0.26      4740

    accuracy                           0.72    291619
   macro avg       0.56      0.77      0.58    291619
weighted avg       0.77      0.72      0.73    291619



In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_sentiment, y_pred)

print(cm)

[[ 87810  24719   3256]
 [ 34994 116598  19502]
 [    84    539   4117]]


##model 2

In [ ]:
model = Sequential([
    Embedding(
        input_dim=embedding_matrix.shape[0],
        output_dim=embedding_matrix.shape[1],
        weights=[embedding_matrix],
        trainable=True
    ),

    SpatialDropout1D(0.2),

    TCN(
        nb_filters=64,
        kernel_size=3,
        dilations=[1,2,4,8],
        dropout_rate=0.2,
        return_sequences=False
    ),

    Dense(32, activation='relu'),

    Dropout(0.5),

    Dense(3, activation='softmax')
])

In [ ]:
model.compile(
    optimizer="adam",
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    X_train_pad,
    y_train_sentiment,
    validation_split=0.1,
    epochs=10,
    batch_size=128,
    class_weight=class_weights,
    callbacks=[early_stopping]
)

Epoch 1/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 88s 9ms/step - accuracy: 0.5516 - loss: 0.7490 - val_accuracy: 0.7337 - val_loss: 0.6044
Epoch 2/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 63s 8ms/step - accuracy: 0.7036 - loss: 0.5393 - val_accuracy: 0.7564 - val_loss: 0.5547
Epoch 3/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 82s 8ms/step - accuracy: 0.7311 - loss: 0.4891 - val_accuracy: 0.7663 - val_loss: 0.5025
Epoch 4/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 62s 7ms/step - accuracy: 0.7472 - loss: 0.4589 - val_accuracy: 0.7584 - val_loss: 0.5302
Epoch 5/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 62s 8ms/step - accuracy: 0.7591 - loss: 0.4394 - val_accuracy: 0.7596 - val_loss: 0.5394


In [ ]:
import numpy as np

y_pred_probs = model.predict(X_test_pad)

y_pred = np.argmax(y_pred_probs, axis=1)

9114/9114 ━━━━━━━━━━━━━━━━━━━━ 22s 2ms/step


In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test_sentiment,
        y_pred,
        target_names=[
            "Negative",
            "Neutral",
            "Positive"
        ]
    )
)

              precision    recall  f1-score   support

    Negative       0.77      0.80      0.79    115785
     Neutral       0.85      0.74      0.79    171094
    Positive       0.18      0.89      0.30      4740

    accuracy                           0.77    291619
   macro avg       0.60      0.81      0.63    291619
weighted avg       0.81      0.77      0.78    291619



In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_sentiment, y_pred)

print(cm)

[[ 92774  21179   1832]
 [ 27218 126874  17002]
 [    40    464   4236]]


##model 3

In [ ]:
class_weights = {
    0: 1.0,
    1: 0.7,
    2: 10.0
}

In [ ]:
model = Sequential([
    Embedding(
        input_dim=embedding_matrix.shape[0],
        output_dim=embedding_matrix.shape[1],
        weights=[embedding_matrix],
        trainable=True
    ),

    SpatialDropout1D(0.2),

    TCN(
        nb_filters=64,
        kernel_size=3,
        dilations=[1,2,4,8],
        dropout_rate=0.2,
        return_sequences=False
    ),

    Dense(32, activation='relu'),

    Dropout(0.5),

    Dense(3, activation='softmax')
])

In [ ]:
model.compile(
    optimizer="adam",
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    X_train_pad,
    y_train_sentiment,
    validation_split=0.1,
    epochs=10,
    batch_size=128,
    class_weight=class_weights,
    callbacks=[early_stopping]
)

Epoch 1/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 84s 9ms/step - accuracy: 0.7061 - loss: 0.6547 - val_accuracy: 0.7857 - val_loss: 0.4714
Epoch 2/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 62s 8ms/step - accuracy: 0.7741 - loss: 0.4964 - val_accuracy: 0.8053 - val_loss: 0.4396
Epoch 3/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 63s 8ms/step - accuracy: 0.7898 - loss: 0.4563 - val_accuracy: 0.8071 - val_loss: 0.4315
Epoch 4/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 62s 8ms/step - accuracy: 0.8008 - loss: 0.4334 - val_accuracy: 0.8092 - val_loss: 0.4271
Epoch 5/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 63s 8ms/step - accuracy: 0.8092 - loss: 0.4163 - val_accuracy: 0.8199 - val_loss: 0.3992
Epoch 6/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 62s 8ms/step - accuracy: 0.8150 - loss: 0.4025 - val_accuracy: 0.8100 - val_loss: 0.4266
Epoch 7/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 65s 8ms/step - accuracy: 0.8196 - loss: 0.3924 - val_accuracy: 0.8122 - val_loss: 0.4213


In [ ]:
import numpy as np

y_pred_probs = model.predict(X_test_pad)

y_pred = np.argmax(y_pred_probs, axis=1)

9114/9114 ━━━━━━━━━━━━━━━━━━━━ 22s 2ms/step


In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test_sentiment,
        y_pred,
        target_names=[
            "Negative",
            "Neutral",
            "Positive"
        ]
    )
)

              precision    recall  f1-score   support

    Negative       0.83      0.79      0.81    115785
     Neutral       0.85      0.84      0.85    171094
    Positive       0.30      0.80      0.43      4740

    accuracy                           0.82    291619
   macro avg       0.66      0.81      0.69    291619
weighted avg       0.83      0.82      0.82    291619



In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_sentiment, y_pred)

print(cm)

[[ 90902  24009    874]
 [ 18990 143980   8124]
 [    37    934   3769]]


##model 4

In [ ]:
class_weights = {
    0: 15,
    1: 4,
    2: 1
}

In [ ]:
model = Sequential([
    Embedding(
        input_dim=embedding_matrix.shape[0],
        output_dim=embedding_matrix.shape[1],
        weights=[embedding_matrix],
        trainable=True
    ),

    SpatialDropout1D(0.2),

    TCN(
        nb_filters=64,
        kernel_size=3,
        dilations=[1,2,4,8],
        dropout_rate=0.2,
        return_sequences=False
    ),

    Dense(32, activation='relu'),

    Dropout(0.5),

    Dense(3, activation='softmax')
])

In [ ]:
model.compile(
    optimizer="adam",
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    X_train_pad,
    y_train_sentiment,
    validation_split=0.1,
    epochs=10,
    batch_size=128,
    class_weight=class_weights,
    callbacks=[early_stopping]
)

Epoch 1/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 95s 10ms/step - accuracy: 0.6928 - loss: 3.4196 - val_accuracy: 0.7836 - val_loss: 0.4970
Epoch 2/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 63s 8ms/step - accuracy: 0.7696 - loss: 2.7159 - val_accuracy: 0.7930 - val_loss: 0.4860


In [ ]:
import numpy as np

y_pred_probs = model.predict(X_test_pad)

y_pred = np.argmax(y_pred_probs, axis=1)

9114/9114 ━━━━━━━━━━━━━━━━━━━━ 21s 2ms/step


In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test_sentiment,
        y_pred,
        target_names=[
            "Negative",
            "Neutral",
            "Positive"
        ]
    )
)

              precision    recall  f1-score   support

    Negative       0.68      0.93      0.79    115785
     Neutral       0.91      0.71      0.79    171094
    Positive       0.00      0.00      0.00      4740

    accuracy                           0.78    291619
   macro avg       0.53      0.55      0.53    291619
weighted avg       0.80      0.78      0.78    291619



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_sentiment, y_pred)

print(cm)

[[107896   7889      0]
 [ 50424 120670      0]
 [   509   4231      0]]
